# 05 - Recognition and Appropriateness Rubric

Weighted target-specific SGD logistic models convert scaled handcrafted features to probabilities with low/high uncertainty thresholds. Every rule is evaluated after a failure so all reasons remain visible.

In [1]:
import csv,sys
from pathlib import Path
import pandas as pd
cwd=Path.cwd().resolve(); REPO=next((p for p in [cwd,*cwd.parents] if (p/'implementation'/'src').exists()),None)
if REPO is None: raise RuntimeError('Start Jupyter inside the repository.')
sys.path.insert(0,str(REPO/'implementation'/'src'))
from ipcv_attire.dataset_policy import load_policy,require_showcase_approval
from ipcv_attire.pipeline import AttirePipeline
from ipcv_attire.training import train_pipeline
DATA=REPO/'implementation'/'data'; POLICY=load_policy(DATA/'dataset-policy.json'); BUNDLE=REPO/'implementation'/'models'/'classical-attire-smoke'
if not (BUNDLE/'manifest.json').exists(): train_pipeline(manifest_dir=DATA/'interim'/'manifests',fashionpedia_root=DATA/'raw'/'fashionpedia',policy=POLICY,bundle_dir=BUNDLE,profile_name='smoke')
pipeline=AttirePipeline.load(BUNDLE)


## Rubric

Immediate failures: revealing or round-neck casual tops, casual/tight or damaged bottoms, leisurewear, and headwear. Required evidence: collar, allowed sleeve, formal-bottom proxy, and footwear. Suspected skorts require review. Tucked-in status, piercing count, exact open-toe subtype, and customary-headgear exceptions are unsupported.

In [2]:
definitions=POLICY['compliance_rules']['rule_definitions']; display(pd.DataFrame([{'rule_id':key,**value} for key,value in definitions.items()]).set_index('rule_id')); print('Unsupported:',POLICY['compliance_rules']['unsupported_conditions']); print('Proxies:',POLICY['compliance_rules']['proxies'])


,label,region,kind,severity,model_supported
rule_id,,,,,
collared_top,Collared top,upper_body,required,conditional,True
allowed_sleeve,Allowed sleeve length,upper_body,required,conditional,True
formal_bottom_candidate,Formal bottom proxy,bottom_body,required,conditional,True
footwear_present,Footwear present,footwear,required,conditional,True
casual_round_neck_top,Round-neck casual top,upper_body,prohibited,immediate,True
revealing_top,Revealing top,upper_body,prohibited,immediate,True
casual_or_tight_bottom,Casual or tight bottom,bottom_body,prohibited,immediate,True
damaged_bottom,Damaged bottom,bottom_body,prohibited,immediate,True
skort_bottom,Skort,bottom_body,prohibited,immediate,False


Unsupported: ['tucked_in_shirt', 'excessive_body_piercings', 'open_toe_footwear_subtype', 'customary_headgear_exception']
Proxies: {'formal_bottom_candidate': 'Pants without jeans or casual-bottom attributes; Fashionpedia has no direct slacks, trousers, or khakis label.', 'footwear_present': 'A shoe instance confirms footwear presence but not whether the footwear is open-toed or otherwise permitted.', 'headwear_present': 'A Fashionpedia hat instance is treated as prohibited headwear because customary use is not annotated.'}


In [3]:
DEMO_IMAGE_ID='13665'
with (DATA/'interim'/'manifests'/'fashionpedia-images.csv').open(encoding='utf-8',newline='') as handle: record=next(r for r in csv.DictReader(handle) if r['image_id']==DEMO_IMAGE_ID)
require_showcase_approval(DEMO_IMAGE_ID,DATA/'showcase-manifest.csv',display_risk=record['display_risk']=='1'); report=pipeline.analyze(DATA/'raw'/'fashionpedia'/record['relative_image_path'])
predictions=[{'outfit':o.outfit_id,'component':c.component_id,**p.__dict__} for o in report.outfits for c in o.components for p in c.predictions]; display(pd.DataFrame(predictions))


,outfit,component,target_id,probability,region,evidence_region,low_threshold,high_threshold
0,outfit-6,component-19,collared_top,0.0,upper_body,component-19,0.10,0.55
1,outfit-6,component-19,allowed_sleeve,1.0,upper_body,component-19,0.10,0.55
2,outfit-6,component-19,casual_round_neck_top,0.0,upper_body,component-19,0.10,0.55
3,outfit-6,component-19,revealing_top,0.0,upper_body,component-19,0.10,0.55
4,outfit-7,component-22,collared_top,1.0,upper_body,component-22,0.10,0.55
5,outfit-7,component-22,allowed_sleeve,0.0,upper_body,component-22,0.10,0.55
6,outfit-7,component-22,casual_round_neck_top,1.0,upper_body,component-22,0.10,0.55
7,outfit-7,component-22,revealing_top,1.0,upper_body,component-22,0.10,0.55
8,outfit-9,component-26,formal_bottom_candidate,0.0,bottom_body,component-26,0.10,0.55
9,outfit-9,component-26,casual_or_tight_bottom,1.0,bottom_body,component-26,0.35,0.55


In [4]:
rules=[{'outfit':o.outfit_id,'decision':o.decision.value,'rule':r.rule_id,'status':r.status.value,'confidence':r.confidence,'evidence':r.evidence_region,'reason':r.reason} for o in report.outfits for r in o.rules]; display(pd.DataFrame(rules)); print('IMAGE DECISION:',report.to_dict()['user_facing_decision'].upper())
for o in report.outfits: print(o.outfit_id,'passed=',o.passed_reasons,'failed=',o.failed_reasons,'review=',o.review_reasons,'unsupported=',o.unsupported_reasons)


,outfit,decision,rule,status,confidence,evidence,reason
0,outfit-1,review_required,collared_top,unknown,NaN,upper_body,Collared top: no reliable model evidence was p...
1,outfit-1,review_required,allowed_sleeve,unknown,NaN,upper_body,Allowed sleeve length: no reliable model evide...
2,outfit-1,review_required,casual_round_neck_top,unknown,NaN,upper_body,Round-neck casual top: no reliable model evide...
3,outfit-1,review_required,revealing_top,unknown,NaN,upper_body,Revealing top: no reliable model evidence was ...
4,outfit-1,review_required,formal_bottom_candidate,unknown,NaN,bottom_body,Formal bottom proxy: no reliable model evidenc...
...,...,...,...,...,...,...,...
141,outfit-9,non_compliant,excessive_body_piercings,unsupported,NaN,NaN,Excessive body piercings are not annotated by ...
142,outfit-9,non_compliant,open_toe_footwear_subtype,unsupported,NaN,NaN,Exact open-toe footwear subtype is not annotat...
143,outfit-9,non_compliant,customary_headgear_exception,unsupported,NaN,NaN,Customary-headgear exceptions are not annotate...
144,outfit-9,non_compliant,technical_review_1,unknown,NaN,NaN,Component-to-outfit grouping is ambiguous.


IMAGE DECISION: INAPPROPRIATE
outfit-1 passed= ['Headwear present passed with confidence 0.000.'] failed= [] review= ['Collared top: no reliable model evidence was produced.', 'Allowed sleeve length: no reliable model evidence was produced.', 'Round-neck casual top: no reliable model evidence was produced.', 'Revealing top: no reliable model evidence was produced.', 'Formal bottom proxy: no reliable model evidence was produced.', 'Casual or tight bottom: no reliable model evidence was produced.', 'Damaged bottom: no reliable model evidence was produced.', 'Leisurewear: no reliable model evidence was produced.', 'Footwear present is uncertain at 0.000; the result remains review-required.', 'Segmentation quality failed for: component-1, component-2.'] unsupported= ['Skort is unsupported by the available class support and was not guessed.', 'Tucked-in shirt status is not annotated by Fashionpedia.', 'Excessive body piercings are not annotated by Fashionpedia.', 'Exact open-toe footwear su

## Precedence

Any confident failure makes an outfit inappropriate. Otherwise incomplete required evidence means review-required. Appropriate requires all supported requirements and no prohibition. Across outfits, failure wins, then review, then appropriate.